# Rainfall Data Analysis — DWD vs. Simulated

**Purpose:** Compares observed daily rainfall from the DWD gauge station
against the precipitation time series used in the J2000 / TALSIM simulation,
and produces summary visualisations.

**What it does:**
- Loads DWD gauge data (Excel) and simulated precipitation (CSV)
- Aligns both series to a common date range (1995-09-01 – 2023-12-31)
- Plots annual rainfall totals, monthly distributions, and scatter comparisons
- Multi-loop version compares across multiple simulation runs

**Input:** DWD station Excel, `Loop_8.csv` (simulated precip)  
**Output:** Rainfall comparison plots

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
file1 = r"C:\Users\raah\Downloads\tageswerte_RR_02651_19470401_20241231_hist (1)\Microsoft Excel-Arbeitsblatt (neu).xlsx"
file2 = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Time_series_data_ZR\Loop_8.csv"


In [ ]:
start_date = '1995-09-01'
end_date = '2023-12-31'

In [ ]:
df1_preview = pd.read_excel(file1, header=None, nrows=10)
header_row_index = None

for i, row in df1_preview.iterrows():
    if 'MESS_DATUM' in row.values:
        header_row_index = i
        break

df1 = pd.read_excel(file1, header=header_row_index)
df1.columns = df1.columns.str.strip()

# Convert MESS_DATUM to proper datetime
df1['DATE'] = pd.to_datetime(df1['MESS_DATUM'], format='%Y%m%d', errors='coerce')

# Clean rainfall: convert + remove error codes
df1['RS'] = pd.to_numeric(df1['RS'], errors='coerce')
df1['RS'] = df1['RS'].replace([-999, -9999], pd.NA)

# Filter for range
df1_filtered = df1[(df1['DATE'] >= start_date) & (df1['DATE'] <= end_date)]
df1_total = df1_filtered['RS'].sum()

print(f"Total rainfall from DWD Daily dataset ({start_date} to {end_date}): {df1_total:.2f} mm")

# -------------------------------------------------------
# ----------- LOAD HOURLY J2K DATA (df2) -----------------
# -------------------------------------------------------

df2 = pd.read_csv(file2, sep=";")
df2.columns = df2.columns.str.strip()

# Find datetime and precipitation column names automatically
datetime_col = [c for c in df2.columns if 'time' in c.lower() or 'datum' in c.lower() or 'date' in c.lower()][0]
precip_col   = [c for c in df2.columns if 'precip' in c.lower() or 'rain' in c.lower()][0]

# Convert to datetime
df2['datetime'] = pd.to_datetime(df2[datetime_col], format='%d.%m.%Y %H:%M', errors='coerce')

# Clean rainfall
df2['precip'] = pd.to_numeric(df2[precip_col], errors='coerce')
df2['precip'] = df2['precip'].replace([-999, -9999], pd.NA)

# Filter
df2_filtered = df2[(df2['datetime'] >= start_date) & (df2['datetime'] <= end_date)]
df2_total = df2_filtered['precip'].sum()

print(f"Total rainfall from J2k Hourly dataset ({start_date} to {end_date}): {df2_total:.2f} mm")

# -------------------------------------------------------
# ------------------- PLOTTING ---------------------------
# -------------------------------------------------------

totals = pd.DataFrame({
    'Source': ['DWD Daily', 'J2k Hourly'],
    'Total Rainfall [mm]': [df1_total, df2_total]
})

sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 6))

colors = ["#4C72B0", "#55A868"]

bars = plt.bar(
    totals['Source'],
    totals['Total Rainfall [mm]'],
    color=colors,
    edgecolor='black',
    linewidth=1.2,
    width=0.4
)

# Add labels above bars
for bar in bars:
    h = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        h + max(totals['Total Rainfall [mm]'])*0.02,
        f'{h:.2f}',
        ha='center',
        va='bottom',
        fontsize=12,
        fontweight='bold'
    )

plt.ylabel('Total Rainfall [mm]', fontsize=12)
plt.title(f'(Loop_8)Total Rainfall Comparison ({start_date} to {end_date})', fontsize=14, fontweight='bold')
plt.ylim(0, max(totals['Total Rainfall [mm]'])*1.25)
plt.grid(axis='y', linestyle='--', alpha=0.7)
sns.despine(left=True, bottom=True)
plt.tight_layout()

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------
# File paths
# -------------------------
file1 = r"C:\Users\raah\Downloads\tageswerte_RR_02651_19470401_20241231_hist (1)\Microsoft Excel-Arbeitsblatt (neu).xlsx"
file2 = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Time_series_data_ZR\Loop_8_modified correction.csv"

# -------------------------
# --- LOAD DAILY DWD FILE ---
# -------------------------

# Detect header row automatically
df1_preview = pd.read_excel(file1, header=None, nrows=15)
header_row = None
for i, row in df1_preview.iterrows():
    if "MESS_DATUM" in row.values:
        header_row = i
        break

df1 = pd.read_excel(file1, header=header_row)
df1.columns = df1.columns.str.strip()

# Convert columns
df1["DATE"] = pd.to_datetime(df1["MESS_DATUM"], format="%Y%m%d", errors="coerce")
df1["RS"] = pd.to_numeric(df1["RS"], errors="coerce")

# Replace -9999 with NaN so they are ignored in sums
df1["RS"] = df1["RS"].replace(-9999, np.nan)

df1["Year"] = df1["DATE"].dt.year

# -------------------------
# --- LOAD HOURLY J2K FILE ---
# -------------------------
df2 = pd.read_csv(file2, sep=";")
df2.columns = df2.columns.str.strip()

# Detect datetime and precip columns
datetime_col = [c for c in df2.columns if "time" in c.lower() or "date" in c.lower()][0]
precip_col   = [c for c in df2.columns if "precip" in c.lower()][0]

# Convert columns
df2["datetime"] = pd.to_datetime(df2[datetime_col], format="%d.%m.%Y %H:%M", errors="coerce")
df2["precip"] = pd.to_numeric(df2[precip_col], errors="coerce")

# Replace -9999 with NaN so they are ignored
df2["precip"] = df2["precip"].replace(-9999, np.nan)

df2["Year"] = df2["datetime"].dt.year

# -------------------------
# --- YEARLY SUM 1996–2023 ---
# -------------------------
years = list(range(2011, 2022))

df1_yearly = df1[df1["Year"].isin(years)].groupby("Year")["RS"].sum().reset_index()
df2_yearly = df2[df2["Year"].isin(years)].groupby("Year")["precip"].sum().reset_index()

# Merge results
yearly_totals = pd.merge(df1_yearly, df2_yearly, on="Year", how="outer")
yearly_totals.rename(columns={"RS": "DWD Daily", "precip": "J2k Hourly"}, inplace=True)

# Replace missing years with zero
yearly_totals.fillna(0, inplace=True)

# -------------------------
# --- MEAN VALUES ---
# -------------------------
mean_daily = yearly_totals["DWD Daily"].mean()
mean_hourly = yearly_totals["J2k Hourly"].mean()

# -------------------------
# --- PLOT ---
# -------------------------
sns.set_theme(style="whitegrid")
plt.figure(figsize=(14, 6))

bar_width = 0.4
x = yearly_totals["Year"]

# Side-by-side bars
plt.bar(x - bar_width/2, yearly_totals["DWD Daily"], width=bar_width, 
        color="#4C72B0", edgecolor="black", label="DWD Daily")

plt.bar(x + bar_width/2, yearly_totals["J2k Hourly"], width=bar_width, 
        color="#55A868", edgecolor="black", label="J2k Hourly")

# Mean lines
plt.axhline(mean_daily, color="#4C72B0", linestyle="--", linewidth=1, label="Mean DWD Daily")
plt.axhline(mean_hourly, color="#55A868", linestyle="--", linewidth=1, label="Mean J2k Hourly")

# Labels
plt.ylabel("Total Rainfall [mm]", fontsize=12)
plt.xlabel("Year", fontsize=12)
plt.title("(Modified_Correction)Total Yearly Rainfall Comparison (1996–2023)", fontsize=15, fontweight="bold")

plt.xticks(years, rotation=45)
plt.ylim(0, yearly_totals[["DWD Daily", "J2k Hourly"]].max().max() * 1.25)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.legend()
sns.despine(left=True, bottom=True)

plt.tight_layout()
plt.show()


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors

# -------------------------
# File paths
# -------------------------
dwd_file = r"C:\Users\raah\Downloads\tageswerte_RR_02651_19470401_20241231_hist (1)\Microsoft Excel-Arbeitsblatt (neu).xlsx"
loop_folder = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Time_series_data_ZR"

# -------------------------
# --- LOAD DWD DAILY DATA ---
# -------------------------
df_preview = pd.read_excel(dwd_file, header=None, nrows=15)
header_row = next((i for i, row in df_preview.iterrows() if "MESS_DATUM" in row.values), None)

df_dwd = pd.read_excel(dwd_file, header=header_row)
df_dwd.columns = df_dwd.columns.str.strip()

df_dwd["DATE"] = pd.to_datetime(df_dwd["MESS_DATUM"], format="%Y%m%d", errors="coerce")
df_dwd["RS"] = pd.to_numeric(df_dwd["RS"], errors="coerce")

# Only valid rainfall: RS >= 0 and not -9999
df_dwd["RS"] = df_dwd["RS"].mask((df_dwd["RS"] < 0) | (df_dwd["RS"] == -9999), pd.NA)
df_dwd["Year"] = df_dwd["DATE"].dt.year

# -------------------------
# --- YEARLY SUM DWD ---
# -------------------------
years = list(range(2011, 2023))
df_dwd_yearly = (
    df_dwd[df_dwd["Year"].isin(years)]
    .groupby("Year")["RS"]
    .sum(min_count=1)
    .reset_index()
    .rename(columns={"RS": "DWD Daily"})
)

# -------------------------
# --- LOAD AND SUM LOOP FILES ---
# -------------------------
loop_files = [f for f in os.listdir(loop_folder) if f.lower().endswith(".csv")]
loop_files.sort()

loop_yearly_dict = {}
for file in loop_files:
    loop_name = os.path.splitext(file)[0]
    path = os.path.join(loop_folder, file)
    
    df = pd.read_csv(path, sep=";")
    df.columns = df.columns.str.strip()
    
    datetime_col = [c for c in df.columns if "time" in c.lower() or "date" in c.lower()][0]
    precip_col = [c for c in df.columns if "precip" in c.lower()][0]
    
    df["datetime"] = pd.to_datetime(df[datetime_col], format="%d.%m.%Y %H:%M", errors="coerce")
    df["precip"] = pd.to_numeric(df[precip_col], errors="coerce")
    
    # Only valid rainfall: >=0 and not -9999
    df["precip"] = df["precip"].mask((df["precip"] < 0) | (df["precip"] == -9999), pd.NA)
    df["Year"] = df["datetime"].dt.year
    
    df_yearly = (
        df[df["Year"].isin(years)]
        .groupby("Year")["precip"]
        .sum(min_count=1)
        .reset_index()
        .rename(columns={"precip": loop_name})
    )
    loop_yearly_dict[loop_name] = df_yearly

# -------------------------
# --- MERGE ALL YEARLY DATA ---
# -------------------------
yearly_totals = df_dwd_yearly.copy()
for loop_name, df_loop in loop_yearly_dict.items():
    yearly_totals = pd.merge(yearly_totals, df_loop, on="Year", how="outer")

yearly_totals.fillna(0, inplace=True)

# -------------------------
# --- PLOT SIDE-BY-SIDE BARS ---
# -------------------------
sns.set_theme(style="whitegrid")
plt.figure(figsize=(18, 8))

num_series = len(yearly_totals.columns) - 1
bar_width = 0.85 / num_series
x = yearly_totals["Year"].values

# Distinct colors for DWD vs loops
colors = ['#1f77b4'] + list(mcolors.TABLEAU_COLORS.values())[1:num_series]

for i, col in enumerate(yearly_totals.columns[1:]):
    plt.bar(
        x + i * bar_width - 0.425 + bar_width/2,
        yearly_totals[col],
        width=bar_width,
        label=col,
        color=colors[i],
       
    )

plt.xlabel("Year", fontsize=14, fontweight="bold")
plt.ylabel("Total Rainfall [mm]", fontsize=14, fontweight="bold")
plt.title("Yearly Rainfall Comparison: DWD vs J2K_Loops (1996–2023)", fontsize=16, fontweight="bold")
plt.xticks(years, rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Legend below the plot
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=min(5, num_series), fontsize=12)

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()


#print(yearly_totals)
